# Create custom exceptions

When your project grows beyond simple scripts, the built-in exception types often lack the specificity you need. A function that raises `ValueError` does not tell the caller whether the problem was an invalid email address, a missing configuration key, or an out-of-range temperature. Custom exceptions solve this by giving each failure mode a distinct type that callers can handle independently.

This guide walks you through building a custom exception hierarchy for your project, from a single exception class to a complete, well-organised module.

## A basic custom exception

The simplest custom exception inherits from `Exception` and does nothing else. This is often all you need to give your code a meaningful, project-specific exception type.

In [ ]:
class PaymentError(Exception):
    """Raised when a payment cannot be processed."""
    pass

You can raise and handle this exception exactly like any built-in exception.

In [ ]:
def process_payment(amount: float) -> str:
    """Process a payment.

    Args:
        amount: The payment amount in pounds.

    Returns:
        A confirmation message.

    Raises:
        PaymentError: If the amount is not positive.
    """
    if amount <= 0:
        raise PaymentError(f"Payment amount must be positive, got {amount}")
    return f"Payment of \u00a3{amount:.2f} processed successfully"


# Successful payment
print(process_payment(49.99))

# Failed payment
try:
    process_payment(-10)
except PaymentError as e:
    print(f"Payment failed: {e}")

**Why this works:** By raising `PaymentError` instead of a generic `ValueError`, the caller can distinguish payment failures from other validation issues and handle each case differently.

## Adding attributes to custom exceptions

A simple message string is not always enough. When the exception handler needs structured information about what went wrong, you can add attributes by defining an `__init__` method.

Always call `super().__init__()` with a human-readable message so that the exception still behaves correctly when printed or logged.

In [ ]:
class ValidationError(Exception):
    """Raised when input validation fails.

    Attributes:
        field: The name of the field that failed validation.
        value: The value that was rejected.
        reason: A description of why validation failed.
    """

    def __init__(self, field: str, value: object, reason: str) -> None:
        self.field = field
        self.value = value
        self.reason = reason
        super().__init__(
            f"Validation failed for '{field}' (value={value!r}): {reason}"
        )

In [ ]:
def validate_age(age: int) -> int:
    """Validate that an age value is within an acceptable range.

    Args:
        age: The age to validate.

    Returns:
        The validated age.

    Raises:
        ValidationError: If the age is negative or unreasonably large.
    """
    if age < 0:
        raise ValidationError("age", age, "must be non-negative")
    if age > 150:
        raise ValidationError("age", age, "must be 150 or less")
    return age


try:
    validate_age(-5)
except ValidationError as e:
    print(f"Error message: {e}")
    print(f"Field: {e.field}")
    print(f"Rejected value: {e.value}")
    print(f"Reason: {e.reason}")

**Why this works:** The exception handler can inspect `e.field`, `e.value`, and `e.reason` directly, rather than parsing the message string. This is especially useful when building error responses for APIs or user interfaces.

## Building an exception hierarchy

As your project grows, you may have several related exception types. Organising them into a hierarchy lets callers choose their level of granularity: they can handle a specific exception or a broad category.

The pattern is straightforward:

1. Define a **base exception** for your project or module
2. Create **specific exceptions** that inherit from the base
3. Callers handle the base to catch everything, or specific subclasses for fine-grained control

In [ ]:
# Base exception for the entire project
class OrderError(Exception):
    """Base exception for order processing failures."""
    pass


# Specific exceptions inherit from the base
class OutOfStockError(OrderError):
    """Raised when a requested item is not in stock.

    Attributes:
        item: The name of the out-of-stock item.
        requested: The quantity requested.
        available: The quantity currently available.
    """

    def __init__(self, item: str, requested: int, available: int) -> None:
        self.item = item
        self.requested = requested
        self.available = available
        super().__init__(
            f"{item}: requested {requested}, but only {available} available"
        )


class InvalidOrderError(OrderError):
    """Raised when an order contains invalid data."""
    pass


class PaymentDeclinedError(OrderError):
    """Raised when a payment method is declined.

    Attributes:
        reason: The reason the payment was declined.
    """

    def __init__(self, reason: str) -> None:
        self.reason = reason
        super().__init__(f"Payment declined: {reason}")

In [ ]:
def place_order(item: str, quantity: int, stock: int) -> str:
    """Place an order for an item.

    Args:
        item: The name of the item to order.
        quantity: The number of items to order.
        stock: The number of items currently in stock.

    Returns:
        A confirmation message.

    Raises:
        InvalidOrderError: If the quantity is not positive.
        OutOfStockError: If there is insufficient stock.
    """
    if quantity <= 0:
        raise InvalidOrderError(f"Quantity must be positive, got {quantity}")
    if quantity > stock:
        raise OutOfStockError(item, quantity, stock)
    return f"Order placed: {quantity}x {item}"

Callers can now handle exceptions at different levels of specificity.

In [ ]:
# Handle a specific exception type
try:
    place_order("Keyboard", 5, 2)
except OutOfStockError as e:
    print(f"Out of stock: {e.item} (wanted {e.requested}, have {e.available})")

In [ ]:
# Handle all order-related exceptions with the base class
orders = [
    ("Monitor", 1, 10),
    ("Mouse", 0, 5),
    ("Webcam", 3, 1),
]

for item, quantity, stock in orders:
    try:
        result = place_order(item, quantity, stock)
        print(result)
    except OrderError as e:
        print(f"Order failed: {e}")

**Why this works:** The `except OrderError` clause handles both `OutOfStockError` and `InvalidOrderError` because they are subclasses of `OrderError`. This follows the same pattern that Python uses with its own built-in exception hierarchy, where handling `OSError` also handles `FileNotFoundError`, `PermissionError`, and other subclasses.

You can verify the hierarchy using `issubclass()`.

In [ ]:
print(f"OutOfStockError is a subclass of OrderError: "
      f"{issubclass(OutOfStockError, OrderError)}")
print(f"OutOfStockError is a subclass of Exception: "
      f"{issubclass(OutOfStockError, Exception)}")
print(f"OrderError is a subclass of OutOfStockError: "
      f"{issubclass(OrderError, OutOfStockError)}")

## Naming conventions

Python has clear conventions for naming exception classes. Following these conventions makes your code immediately recognisable to other Python developers.

**Use the `Error` suffix.** Almost all exception classes in the standard library end with `Error`: `ValueError`, `TypeError`, `FileNotFoundError`, and so on. Your custom exceptions should follow the same convention.

| Good | Avoid |
|------|-------|
| `InsufficientFundsError` | `InsufficientFunds` |
| `ConfigurationError` | `ConfigException` |
| `AuthenticationError` | `AuthFailed` |
| `RateLimitError` | `TooManyRequests` |

**Use descriptive names.** The exception name should clearly communicate what went wrong, without needing to read the message.

| Good | Avoid |
|------|-------|
| `InvalidEmailError` | `EmailError` |
| `DuplicateEntryError` | `DataError` |
| `ConnectionTimeoutError` | `TimeoutError` (conflicts with built-in) |

**Avoid shadowing built-in names.** Do not name your exception `ValueError`, `TypeError`, or any other built-in exception name. If you need a more specific version, prefix it with your domain.

In [ ]:
# Good: clearly named, descriptive, uses Error suffix
class InvalidPostcodeError(Exception):
    """Raised when a postcode does not match the expected format."""
    pass


class DuplicateUsernameError(Exception):
    """Raised when attempting to register an already-taken username."""
    pass


class InsufficientPermissionsError(Exception):
    """Raised when the user lacks the required permissions."""
    pass


print("All exception classes defined successfully.")

## Organising exceptions into a module

For larger projects, define all your custom exceptions in a dedicated module. This keeps them in one place and makes imports clean.

A common pattern is to create an `exceptions.py` file (or `errors.py`) at the top level of your package.

```
myproject/
    __init__.py
    exceptions.py     # All custom exceptions live here
    orders.py
    payments.py
    users.py
```

The `exceptions.py` module would contain the following:

```python
"""Custom exceptions for the myproject package."""


class MyProjectError(Exception):
    """Base exception for all myproject exceptions."""
    pass


# Order-related exceptions

class OrderError(MyProjectError):
    """Base exception for order processing failures."""
    pass


class OutOfStockError(OrderError):
    """Raised when a requested item is not in stock."""

    def __init__(self, item: str, requested: int, available: int) -> None:
        self.item = item
        self.requested = requested
        self.available = available
        super().__init__(
            f"{item}: requested {requested}, but only {available} available"
        )


class InvalidOrderError(OrderError):
    """Raised when an order contains invalid data."""
    pass


# Payment-related exceptions

class PaymentError(MyProjectError):
    """Base exception for payment failures."""
    pass


class PaymentDeclinedError(PaymentError):
    """Raised when a payment method is declined."""

    def __init__(self, reason: str) -> None:
        self.reason = reason
        super().__init__(f"Payment declined: {reason}")


class InsufficientFundsError(PaymentError):
    """Raised when the account balance is too low."""

    def __init__(self, required: float, available: float) -> None:
        self.required = required
        self.available = available
        super().__init__(
            f"Insufficient funds: required \u00a3{required:.2f}, "
            f"available \u00a3{available:.2f}"
        )
```

Other modules then import what they need:

```python
from myproject.exceptions import OutOfStockError, InsufficientFundsError
```

This approach has several advantages:

- All exceptions are defined in one place, making them easy to find and maintain
- Other modules import only the exceptions they need
- The hierarchy is visible at a glance by reading a single file
- External users of your package can import exceptions from a predictable location

## Alternative approaches

### Using `__str__` for custom formatting

Instead of building the message in `__init__` and passing it to `super().__init__()`, you can override `__str__` to control how the exception is displayed. This can be useful when the exception stores structured data and you want the display format to be separate from the data.

In [ ]:
class TemperatureRangeError(Exception):
    """Raised when a temperature is outside the acceptable range.

    Attributes:
        temperature: The temperature value that was rejected.
        min_temp: The minimum acceptable temperature.
        max_temp: The maximum acceptable temperature.
    """

    def __init__(
        self,
        temperature: float,
        min_temp: float,
        max_temp: float,
    ) -> None:
        self.temperature = temperature
        self.min_temp = min_temp
        self.max_temp = max_temp
        super().__init__(str(self))

    def __str__(self) -> str:
        return (
            f"Temperature {self.temperature}\u00b0C is outside the acceptable "
            f"range of {self.min_temp}\u00b0C to {self.max_temp}\u00b0C"
        )


try:
    raise TemperatureRangeError(temperature=105.0, min_temp=0.0, max_temp=100.0)
except TemperatureRangeError as e:
    print(e)
    print(f"Received: {e.temperature}\u00b0C")
    print(f"Acceptable range: {e.min_temp}\u00b0C to {e.max_temp}\u00b0C")

### Using dataclass-style exceptions

For exceptions with many attributes, you can keep the `__init__` concise by following a consistent pattern. This approach is not a true dataclass (exceptions do not work well with `@dataclass`), but it borrows the clarity of that style.

In [ ]:
class HttpError(Exception):
    """Raised when an HTTP request fails.

    Attributes:
        status_code: The HTTP status code.
        url: The URL that was requested.
        method: The HTTP method used.
        body: The response body, if available.
    """

    def __init__(
        self,
        status_code: int,
        url: str,
        method: str = "GET",
        body: str | None = None,
    ) -> None:
        self.status_code = status_code
        self.url = url
        self.method = method
        self.body = body
        super().__init__(f"{method} {url} returned {status_code}")


try:
    raise HttpError(404, "https://api.example.com/users/42")
except HttpError as e:
    print(f"Request failed: {e}")
    print(f"Status code: {e.status_code}")
    print(f"URL: {e.url}")

## Complete working example

The following example brings together all the patterns from this guide. It models a simple library system with a full exception hierarchy, descriptive names, custom attributes, and clean organisation.

In [ ]:
# --- Exception hierarchy ---

class LibraryError(Exception):
    """Base exception for all library system exceptions."""
    pass


class BookNotFoundError(LibraryError):
    """Raised when a book is not found in the catalogue.

    Attributes:
        isbn: The ISBN that was looked up.
    """

    def __init__(self, isbn: str) -> None:
        self.isbn = isbn
        super().__init__(f"No book found with ISBN {isbn}")


class BookNotAvailableError(LibraryError):
    """Raised when a book exists but is not available for borrowing.

    Attributes:
        isbn: The ISBN of the book.
        title: The title of the book.
    """

    def __init__(self, isbn: str, title: str) -> None:
        self.isbn = isbn
        self.title = title
        super().__init__(f"'{title}' (ISBN {isbn}) is currently on loan")


class BorrowingLimitError(LibraryError):
    """Raised when a member has reached their borrowing limit.

    Attributes:
        member_id: The member's identifier.
        limit: The maximum number of books allowed.
    """

    def __init__(self, member_id: str, limit: int) -> None:
        self.member_id = member_id
        self.limit = limit
        super().__init__(
            f"Member {member_id} has reached the borrowing limit of {limit} books"
        )

In [ ]:
# --- Library system ---

class Library:
    """A simple library system for managing book loans."""

    BORROWING_LIMIT = 3

    def __init__(self) -> None:
        # Catalogue: ISBN -> {title, available}
        self._catalogue: dict[str, dict[str, str | bool]] = {}
        # Member loans: member_id -> list of ISBNs
        self._loans: dict[str, list[str]] = {}

    def add_book(self, isbn: str, title: str) -> None:
        """Add a book to the catalogue.

        Args:
            isbn: The book's ISBN.
            title: The book's title.
        """
        self._catalogue[isbn] = {"title": title, "available": True}

    def borrow_book(self, isbn: str, member_id: str) -> str:
        """Borrow a book from the library.

        Args:
            isbn: The ISBN of the book to borrow.
            member_id: The borrowing member's identifier.

        Returns:
            A confirmation message.

        Raises:
            BookNotFoundError: If the ISBN is not in the catalogue.
            BookNotAvailableError: If the book is currently on loan.
            BorrowingLimitError: If the member has reached the borrowing limit.
        """
        # Check if book exists
        if isbn not in self._catalogue:
            raise BookNotFoundError(isbn)

        book = self._catalogue[isbn]
        title = str(book["title"])

        # Check availability
        if not book["available"]:
            raise BookNotAvailableError(isbn, title)

        # Check borrowing limit
        member_loans = self._loans.get(member_id, [])
        if len(member_loans) >= self.BORROWING_LIMIT:
            raise BorrowingLimitError(member_id, self.BORROWING_LIMIT)

        # Process the loan
        book["available"] = False
        self._loans.setdefault(member_id, []).append(isbn)
        return f"'{title}' loaned to member {member_id}"

In [ ]:
# Set up the library
library = Library()
library.add_book("978-0-13-110362-7", "The C Programming Language")
library.add_book("978-0-201-63361-0", "Design Patterns")
library.add_book("978-0-596-00712-6", "Head First Design Patterns")
library.add_book("978-0-13-468599-1", "The Pragmatic Programmer")

# Demonstrate different exception types
test_cases = [
    ("978-0-13-110362-7", "M001", "Borrow an available book"),
    ("978-0-13-110362-7", "M002", "Borrow a book that is on loan"),
    ("978-0-00-000000-0", "M001", "Borrow a book that does not exist"),
    ("978-0-201-63361-0", "M001", "Second loan for M001"),
    ("978-0-596-00712-6", "M001", "Third loan for M001"),
    ("978-0-13-468599-1", "M001", "Fourth loan (exceeds limit)"),
]

for isbn, member, description in test_cases:
    print(f"\n--- {description} ---")
    try:
        result = library.borrow_book(isbn, member)
        print(f"Success: {result}")
    except BookNotFoundError as e:
        print(f"Not found: {e} (ISBN: {e.isbn})")
    except BookNotAvailableError as e:
        print(f"Unavailable: {e}")
    except BorrowingLimitError as e:
        print(f"Limit reached: {e} (limit: {e.limit})")

Each exception type carries the information the caller needs to respond appropriately. The handler for `BookNotFoundError` can access the ISBN, the handler for `BookNotAvailableError` knows the title, and the handler for `BorrowingLimitError` knows the limit. A caller that does not need fine-grained handling can simply catch `LibraryError` to handle all library-related failures in one place.

In [ ]:
# Handling all library exceptions with the base class
try:
    library.borrow_book("978-0-00-000000-0", "M003")
except LibraryError as e:
    print(f"Library operation failed: {e}")
    print(f"Exception type: {type(e).__name__}")

## Summary

This guide covered the following patterns for creating custom exceptions:

- **Basic custom exceptions** inherit from `Exception` with a docstring and `pass`
- **Attributes** are added through a custom `__init__` that calls `super().__init__()` with a readable message
- **Exception hierarchies** use a base exception for the project, with specific subclasses for each failure mode
- **Naming conventions** require an `Error` suffix and descriptive names that do not shadow built-in exceptions
- **Module organisation** keeps all exceptions in a single `exceptions.py` file for discoverability

When designing your exceptions, start simple. A basic custom exception with a good name and a clear message is far better than a complex hierarchy you do not need yet. Add attributes and subclasses as your project demands them.